# 04 · Full Search Walkthrough

An end-to-end demonstration of the `RacquetSearchEngine` — from setup through a
natural-language query to ranked results.

This is the same engine the FastAPI app serves; this notebook exercises it directly
so the retrieval behaviour can be inspected outside the web layer. It shows the full
hybrid pipeline: LLM query parsing, semantic (dense) retrieval, BM25 (lexical)
retrieval, and Reciprocal Rank Fusion (RRF) combining the two into a single ranking.


## Setup

In [1]:
import logging

import pandas as pd

from frame_finder.engine import RacquetSearchEngine
from frame_finder.config import EMBEDDING_MODEL_NAME
from frame_finder.adapters import AnthropicAdapter

pd.set_option("display.max_colwidth", None)

# INFO logging surfaces the engine's internal steps (parsing status, retrieval, fusion).
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
)


/Users/rohankrishnan/Documents/GitHub/racquet-sem-search/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We construct the engine with an LLM adapter for query parsing, the embedding model
name, and the path to the processed serving artifacts (the racquet data CSV and the
precomputed embeddings). `setup()` performs the expensive one-time initialisation:
instantiating the embedder, loading the embeddings, building the BM25 corpus, and
indexing the BM25 retriever.


In [2]:
anthropic_adapter = AnthropicAdapter()

engine = RacquetSearchEngine(
    path_to_artifacts="../data/processed",
    embedder_name=EMBEDDING_MODEL_NAME,
    llm_adapter=anthropic_adapter,
)

engine.setup()


2026-07-03 03:32:18,270 — INFO — Instantiating sentence-transformers/all-MiniLM-L6-v2...
2026-07-03 03:32:18,308 — INFO — No device provided, using mps
2026-07-03 03:32:18,403 — INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-03 03:32:18,403 — WARNING — Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-03 03:32:18,417 — INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-07-03 03:32:18,443 — INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-03 03:32:18,460 — INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/m

## A natural-language query

The core value of Frame Finder is that a user can describe their game in plain
language rather than filtering on specs. The engine parses that description into a
semantic query and a keyword query, runs both retrieval methods, and fuses the
results.


In [3]:
query = "Arm-friendly racquet with easy power that is good for beginners."

result = engine.search(query=query)


2026-07-03 03:32:27,369 — INFO — HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.36it/s]


`search()` returns a `SearchResult`. Its `parsing_status` records whether the LLM
successfully parsed the query, and `fused_result` is the ranked list of
`(racquet_id, rrf_score)` tuples after Reciprocal Rank Fusion.


In [4]:
print("Parsing status:", result.parsing_status)
print("Parsed query:", result.parsed_query)
print()
print("Top 10 fused results (racquet_id, RRF score):")
for racquet_id, score in result.fused_result[:10]:
    print(f"  {racquet_id:>16}   {score:.5f}")


Parsing status: ParsingStatus.SUCCESS
Parsed query: semantic_query='arm-friendly racquet with easy power that is good for beginners' keyword_query='High Slow'

Top 10 fused results (racquet_id, RRF score):
             LX800   0.03089
             WTRI3   0.02848
              TIS6   0.02686
              GXSS   0.02623
            HBMPL6   0.02496
            BMPLA6   0.02372
             VOSV2   0.02341
            Q20KIG   0.02212
            LX1000   0.02133
            GXSMPR   0.02056


## Inspecting the top results

Joining the ranked IDs back to the racquet data shows what was actually surfaced.
For an arm-friendly, easy-power, beginner query we expect to see flexible (low
stiffness), powerful, lighter frames near the top.


In [5]:
top_ids = [racquet_id for racquet_id, _ in result.fused_result[:10]]

top_df = (
    engine.racquet_df.set_index("racquet_id")
    .loc[top_ids]
    .reset_index()
)

top_df[[
    "racquet_id",
    "racquet_name",
    "racquet_power_level",
    "racquet_swing_speed",
    "racquet_price",
]]


,racquet_id,racquet_name,racquet_power_level,racquet_swing_speed,racquet_price
0,LX800,Dunlop LX 800,Medium-High,Slow-Moderate,249.99
1,WTRI3,Wilson Triad Three,Medium-High,Slow-Moderate,269.00
2,TIS6,Head Titanium Ti.S6 Strung,High,Slow-Moderate,99.00
3,GXSS,Head Graphene XT Speed S,Low-Medium,Medium-Fast,99.00
4,HBMPL6,Head Boom MP L 2026,Low-Medium,Medium-Fast,249.00
5,BMPLA6,Head Boom MP L 2026 Purple,Low-Medium,Medium-Fast,249.00
6,VOSV2,Volkl Vostra V2,Medium-High,Slow-Moderate,219.99
7,Q20KIG,ProKennex Ki Q+ 20 (285g),Medium,Medium,249.00
8,LX1000,Dunlop LX 1000,High,Slow,249.99
9,GXSMPR,Head Graphene XT Speed MP,Low-Medium,Fast,99.00


### The single best match

The top-ranked frame, with its distilled description, to sanity-check that the
retrieval matched the intent of the query.


In [6]:
top_id = result.fused_result[0][0]
best = engine.racquet_df[engine.racquet_df["racquet_id"] == top_id].iloc[0]

print(best["racquet_name"])
print("-" * 80)
print(best["distilled_description"])


Dunlop LX 800
--------------------------------------------------------------------------------
User-friendly, arm-friendly racquet under 10 ounces with generous 110 sq in head for beginner and early intermediate players. Fast and explosive at standard 27 inches with easy power and spin generation. Straight String System and Dual Bridge combine comfort and power with responsive sweet spot.


## Trying another query

A different query with a very different intended player, to confirm the rankings
shift accordingly rather than returning the same frames regardless of input.


In [7]:
query_2 = "A control-oriented player's racquet for an advanced baseliner with a fast, flat swing."

result_2 = engine.search(query=query_2)

top_ids_2 = [racquet_id for racquet_id, _ in result_2.fused_result[:10]]
engine.racquet_df.set_index("racquet_id").loc[top_ids_2].reset_index()[[
    "racquet_name",
    "racquet_power_level",
    "racquet_swing_speed",
    "racquet_price",
]]


2026-07-03 03:32:47,440 — INFO — HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 14.15it/s]


,racquet_name,racquet_power_level,racquet_swing_speed,racquet_price
0,Volkl V-Cell V1 Pro,Low-Medium,Medium-Fast,129.00
1,Yonex VCORE 95,Low,Fast,149.00
2,Volkl V-Cell V8 Pro 2023,Low,Fast,119.00
3,Volkl V8 Pro v3,Low,Fast,249.99
4,Head Graphene XT Speed S,Low-Medium,Medium-Fast,99.00
5,Volkl Vostra V9 305g,Low-Medium,Medium-Fast,249.99
6,Volkl Vostra V1 Pro,Low-Medium,Medium-Fast,269.99
7,Dunlop FX 500 Lite,Low-Medium,Medium-Fast,149.00
8,Volkl Vostra V7,Low-Medium,Medium-Fast,239.99
9,Yonex EZONE 98 (2025),Low-Medium,Medium-Fast,305.00


## Summary

The engine takes a free-text description, parses it into semantic and keyword
components, retrieves candidates through both dense and lexical methods, and fuses
them via RRF into a single ranking. The two contrasting queries above return
appropriately different frames, confirming the retrieval responds to the intent of
the query rather than returning a fixed list.

This is the same `search()` call the FastAPI `/search` route wraps — see `app/routers/search.py`.
